# 04 — Tables and GIS (the record; slimmed 2026-09-28)

Read-only on the run. Writes the things worth keeping that are not slides: the link-class record
(the eight-cell truth table, the floor effect, the class counts), the CSV tables **T0** node key ·
**T3** every link · **T2** links with no affordable alternative · **T4** links already connected within
protected land · **T1** the examples, the **GIS export** (pressure classes, refugia, nodes, examples as
GeoPackages + `style.json` for QGIS), and one cell of numbers for `spec/results_log.md`. Everything
lands in `<run>/director_package/{tables,gis}`. Minutes, not hours.

**No figures here** (Ethan 2026-09-28): the Y2Y-scale contract figures (W0–W4, the engine record maps)
were never going to be shared; the functions stay in `wolverine_director` / `corridors_core` and the
last cell shows how to draw one on demand. The presentation set is **`06_director_outputs.ipynb`**,
which reads the run + the 05 product — nothing in 06 depends on this notebook.

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

import matplotlib.pyplot as plt
import corridors_mapstyle as ms
import corridors_director as cd
import director_core as dc
import wolverine_director as wd
for _m in (ms, cd, dc, wd):
    importlib.reload(_m)

In [ ]:
RUN = "v2_run001"
R = cc.load_results(config.RESULTS_DIR / CFG["results_subdir"] / RUN)
P = wd.package(R)
assert P.classified, "this run was classified before D23-D25 -- re-run notebook 03 (counterfactual -> branches -> classify)"

## 1 · The link-class record — the eight-cell table (G18), the floor effect (G19), the counts

In [ ]:
tt = pd.read_csv(R.run_dir / "class_truth_table.csv"); fe = pd.read_csv(R.run_dir / "floor_effect.csv")
print("eight-cell table (E = no alternative link, B1 = one branch, S = narrow):"); print(tt.to_string(index=False))
print("\nfloor effect (registered vs halved / doubled floors):"); print(fe.to_string(index=False))
R.edges["link_class_label"].value_counts()

## 2 · Tables — T0 node key · T3 every link · T2 no affordable alternative · T4 already connected within protected land · T1 examples

In [ ]:
t0 = wd.table_nodes(P)
t3 = wd.table_links(P, "all"); t2 = wd.table_links(P, "flagged"); t4 = wd.table_links(P, "satisfied"); t1 = wd.table_links(P, "examples")
t1

## 3 · GIS export (QGIS: `gis/style.json` names the fields and colours)

In [ ]:
wd.export_gis(P)

## 4 · Numbers for `spec/results_log.md` (R5.2 / R5.3 / R6)

In [ ]:
e = R.edges[(R.edges["cost"] > 0) & (~R.edges["is_adjacency"])]
km2 = R.cell_km2
print(f"{R.run_id}: {len(e)} links between {len(P.nodes)} nodes | band = {R.cutoff:.4f} cost units = about {R.cutoff*R.cell_km:.1f} km of extra travel on open ground (D31)")
print("classes (link_class):"); print(e["link_class"].value_counts().to_string())
for col, lab in (("near_contiguous", "near-contiguous (D25)"), ("width_not_assessable", "width not assessable (D24)"),
                 ("edge_irreplaceable", "no affordable alternative (D7)"), ("route_irreplaceable", "route-irreplaceable as amended (one branch AND narrow)"),
                 ("route_irreplaceable_topo", "one branch only (the retired topology flag)"), ("squeezed", "narrowing (squeeze_ratio_obs < 0.5)")):
    if col in e.columns:
        print(f"  {lab:60s} {int(e[col].fillna(False).astype(bool).sum())}")
if "alt_kind" in e.columns:
    print("alternative-link kind among links WITH an affordable alternative (D29):"); print(e.loc[~e["edge_irreplaceable"].astype(bool), "alt_kind"].value_counts(dropna=False).to_string())
for col in ("open_ground_width_med", "lcp_len_cells", "squeeze_ratio_obs", "width_ratio_p10", "n_branches"):
    if col in e.columns:
        print(f"{col}: " + e[col].describe()[["min", "25%", "50%", "75%", "max"]].round(3).to_string().replace("\n", " · "))
n_pa, n_ip = int((P.secured_by == "pa").sum()), int((P.secured_by == "ipca").sum())
print(f"W11: {n_pa} links already connected within existing PAs, {n_ip} only once the proposed IPCAs are realized; "
      f"corridor land {int(R.corridor.sum())*km2:,.0f} km², outside PAs + IPCAs {int(P.corridor_unprotected.sum())*km2:,.0f} km²")
print("examples (class-and-width-first):", [(x["num"], x["edge_id"], x["act"], x["kind"]) for x in P.examples])

## 5 · A contract figure on demand (not run)

The Y2Y-scale figures still exist as functions — uncomment ONE line if you ever need a record figure.
Each takes minutes (windowed hillshade + 130 node chips + 170 link labels).

In [ ]:
# wd.check_palette(); wd.figure_w1(P)          # corridor pressure, whole Y2Y (TALL)
# wd.figure_act(P, "north")                    # one act crop: "north" | "central" | "south"
# wd.figure_w0(P); wd.figure_w0b(P); wd.figure_w0c(P)
# cc.branches_map(R); cc.near_opt_tiers_map(R)  # engine record views, Y2Y-wide